# 먹을 결심 🫪👅🧑‍🩰🐽🍤 — v4
냉장고 재료로 만드는 다이어트·단백질 2끼/3끼 식단.

프로젝트 폴더에서 `uv sync` → `uv run jupyter lab`. VS Code에서는 프로젝트 `.venv` 커널을 선택하세요.
실행에는 이 노트북과 공통 코드 폴더가 함께 필요합니다.

In [ ]:
from pathlib import Path
import shutil
ROOT=Path.cwd()
if not (ROOT/'.env').exists():shutil.copyfile(ROOT/'.env.example',ROOT/'.env')
print('.env 준비 완료. 세 변수만 사용합니다. 실제 키는 .env에서 편집하세요.')


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists(): raise RuntimeError('프로젝트 폴더에서 노트북을 열어주세요.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from fridge_protein.config import load_settings,TITLE
print(TITLE)
print(load_settings())  # 비밀키 값은 출력하지 않습니다.


## 키 없이 실제 출처 레시피로 계산 확인
영양 수치는 **샘플**, 레시피는 실제 만개의레시피에서 확인한 재료 사실을 사용합니다.
소금·기름 보유를 켠 예시입니다. 2끼와 3끼 모두 메뉴 생성과 하루 재고 보존을 확인합니다.
부족량이 남을 수 있으며 만들어 낸 수치로 목표를 채우지 않습니다.

In [ ]:
from fridge_protein.simple_ui import generate,DEFAULT_SELECTED
from fridge_protein.food_picker import selection_rows
for n in (2,3):
    values=[60,'근육 증가 목표',n,DEFAULT_SELECTED,selection_rows(DEFAULT_SELECTED),True,50,20,20,4,'',False,'sample']
    result,session,detail=generate(*values,None)
    if 'error' in result:raise RuntimeError(result['error'])
    p=result['plan']
    print(n,'끼 /',p['kind'],'/ 단백질',round(p['protein_g'],2),'g / 부족',round(p['deficit_g'],2),'g')
    for m in p['meals']:print('  ',m['title'],round(m['protein_g'],2),'g',m['url'])
    assert p['kind']=='recipe_meals'
    assert all(x['used_g']<=x['available_g']+1e-7 for x in p['leftovers'])


## 웹 화면 열기
재료를 모두 선택 → 보유량 수정 → 소금·기름 보유 확인 → 추천받기.
API 연결·개당 중량·AI 옵션은 설정에 모았습니다. 입력을 바꾸면 이전 결과가 지워집니다.
기존 7860 서버가 실행 중이면 먼저 해당 터미널에서 Ctrl+C로 종료하세요.

In [ ]:
from fridge_protein.simple_ui import build_app,CSS
demo=build_app()
demo.launch(server_name='127.0.0.1',share=False,inbrowser=False,inline=True,css=CSS)


In [ ]:
# 종료할 때 실행: demo.close()